<a href="https://colab.research.google.com/github/Vivekshrotriya1/18_April_Capstone_project/blob/main/18_April_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install azure-ai-documentintelligence

from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
from google.colab import userdata

In [24]:


from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

ENDPOINT = "https://viv54321.cognitiveservices.azure.com/"
KEY = userdata.get('task1')

def analyze_insurance_claim(file_path):
    client = DocumentIntelligenceClient(
        endpoint=ENDPOINT,
        credential=AzureKeyCredential(KEY)
    )

    with open(file_path, "rb") as f:
        file_content = f.read()

    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=AnalyzeDocumentRequest(bytes_source=file_content),
        features=["keyValuePairs"]
    )

    result = poller.result()

    extracted_data = {}

    if result.key_value_pairs:
        for pair in result.key_value_pairs:
            key = pair.key.content if pair.key else "Unknown"
            value = pair.value.content if pair.value else "N/A"
            # print(f"{key} : {value}")
            extracted_data[key] = value

    return extracted_data

result = analyze_insurance_claim(file_name)
# print(result)


important_fields = ["Customer_ID", "Claim_Amount", "Claim_Type", "Location", "Previous_Claims"]

filtered = {}

for k, v in result.items():
    # Clean the key: remove colons, extra spaces, lowercase
    k_cleaned = k.strip().rstrip(":").replace(" ", "_").lower()

    for field in important_fields:
        field_cleaned = field.lower()

        if field_cleaned in k_cleaned or k_cleaned in field_cleaned:
            filtered[field] = v  # Store with your standardized key name
            break

print(filtered)
print(filtered.keys())

{'Customer_ID': 'C001', 'Claim_Type': 'Medical', 'Location': 'Delhi', 'Claim_Amount': '75000', 'Previous_Claims': '2'}
dict_keys(['Customer_ID', 'Claim_Type', 'Location', 'Claim_Amount', 'Previous_Claims'])


In [25]:
import urllib.request
import json

# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script
data = {"Inputs": {"input1": [filtered]}, "GlobalParameters": {}}

body = str.encode(json.dumps(data))

url = 'http://dd50616c-636e-468a-b359-54e18c6c4d9e.koreacentral.azurecontainer.io/score'
# Replace this with the primary/secondary key, AMLToken, or Microsoft Entra ID token for the endpoint
api_key = userdata.get('model')
if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")


headers = {'Content-Type':'application/json', 'Accept': 'application/json', 'Authorization':('Bearer '+ api_key)}

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'{"Results": {"WebServiceOutput0": [{"Customer_ID": "C001", "Claim_Amount": 75000, "Claim_Type": "Medical", "Location": "Delhi", "Previous_Claims": 2, "Scored Labels": false, "Scored Probabilities": 0.2886965166571553}]}}'


In [26]:
import requests

# 1. Configuration
search_url = "https://document-ai-search-ayan.search.windows.net/indexes/claims-index/docs/search?api-version=2021-04-30-Preview"
search_key = userdata.get('search')

headers = {
    "Content-Type": "application/json",
    "api-key": search_key
}

def run_query(search_text, filter_expr=None):
    query = {"search": search_text}
    if filter_expr:
        query["filter"] = filter_expr

    response = requests.post(search_url, headers=headers, json=query)

    if response.status_code == 200:
        data = response.json()
        print(f"\n--- Results for '{search_text}' ---")
        for result in data.get("value", []):
            print(f"Name: {result.get('name')} | Amount: {result.get('claim_amount')} | Location: {result.get('location')}")
    else:
        print(f"Error {response.status_code}: {response.text}")

# 🎯 TASK A: Basic Search
run_query("Rahul Sharma")

# 🎯 TASK B: Search "Claims above 50,000"
run_query("*", filter_expr="claim_amount gt 50000")

# 🎯 TASK C: Search "Fraud cases in Delhi"
run_query("fraud", filter_expr="location eq 'Delhi'")


--- Results for 'Rahul Sharma' ---
Name: Rahul Sharma | Amount: 75000 | Location: Delhi

--- Results for '*' ---
Name: Rahul Sharma | Amount: 75000 | Location: Delhi

--- Results for 'fraud' ---
